# 12 — Capstone: Production LLM Stack

## Background

Individual notebooks covered MCP servers, gateways, caching, prompt management, structured output, multi-modal routing, and observability. This capstone wires them into a single coherent production stack and runs an end-to-end stress test.

## What You'll Learn

- How a request traverses: client -> gateway -> cache -> prompt compiler -> LLM -> observability
- Rate limiting and circuit breaking wired into the gateway
- Prompt versioning with A/B slot assignment
- Cache hit/miss metrics feeding back into cost dashboards
- Failure injection to validate fallback chains
- A metrics report suitable for a production runbook

## Why This Matters

A production LLM system is only as good as its worst failure mode. Understanding how every component interacts under load — and how gracefully it degrades — is the difference between a demo and a product.

In [ ]:
import time, uuid, json, hashlib, random, statistics
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from collections import deque, defaultdict
from enum import Enum
import numpy as np

# ── 1. Prompt versioning ───────────────────────────────────────────────────
@dataclass
class PromptVersion:
    version_id:  str
    template:    str
    slot_weight: float

    def render(self, **kwargs) -> str:
        result = self.template
        for k, v in kwargs.items():
            result = result.replace('{' + k + '}', str(v))
        return result

class PromptVersionManager:
    def __init__(self): self._registry: Dict[str, List[PromptVersion]] = {}

    def register(self, name: str, version: PromptVersion):
        self._registry.setdefault(name, []).append(version)

    def select(self, name: str) -> PromptVersion:
        versions = self._registry.get(name, [])
        if not versions: raise KeyError(f'No prompt: {name}')
        weights = [v.slot_weight for v in versions]
        total   = sum(weights)
        r       = random.random() * total
        accum   = 0.0
        for v in versions:
            accum += v.slot_weight
            if r <= accum: return v
        return versions[-1]

pvm = PromptVersionManager()
pvm.register('summarize', PromptVersion('v1', 'Summarize: {text}', 0.5))
pvm.register('summarize', PromptVersion('v2', 'Key points from: {text}', 0.5))

counts = {'v1': 0, 'v2': 0}
for _ in range(200): counts[pvm.select('summarize').version_id] += 1
print(f'A/B split (200 draws): {counts}')


In [ ]:
# ── 2. Semantic cache ──────────────────────────────────────────────────────
@dataclass
class CacheEntry:
    key:           str
    response:      str
    model:         str
    input_tokens:  int
    output_tokens: int
    created_at:    float = field(default_factory=time.time)
    ttl_s:         float = 300.0
    hits:          int   = 0

    def is_expired(self) -> bool: return (time.time() - self.created_at) > self.ttl_s

class LLMCache:
    def __init__(self, max_size: int = 1000, ttl_s: float = 300.0):
        self._store: Dict[str, CacheEntry] = {}
        self.max_size = max_size
        self.ttl_s    = ttl_s
        self.hits     = 0
        self.misses   = 0

    def _key(self, prompt, model, **params) -> str:
        raw = json.dumps({'prompt': prompt, 'model': model, **params}, sort_keys=True)
        return hashlib.sha256(raw.encode()).hexdigest()[:32]

    def get(self, prompt, model, **params) -> Optional[CacheEntry]:
        key   = self._key(prompt, model, **params)
        entry = self._store.get(key)
        if entry is None or entry.is_expired():
            if entry: del self._store[key]
            self.misses += 1
            return None
        entry.hits += 1
        self.hits  += 1
        return entry

    def put(self, prompt, model, response, in_tok, out_tok, **params):
        if len(self._store) >= self.max_size:
            oldest = min(self._store.values(), key=lambda e: e.created_at)
            del self._store[oldest.key]
        key = self._key(prompt, model, **params)
        self._store[key] = CacheEntry(key, response, model, in_tok, out_tok, ttl_s=self.ttl_s)

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total else 0.0

print('Cache defined.')


In [ ]:
# ── 3. Rate limiter + circuit breaker ─────────────────────────────────────
class RateLimiter:
    def __init__(self, rpm: int = 60, tpm: int = 100_000):
        self.rpm = rpm; self.tpm = tpm
        self._req_ts:   deque = deque()
        self._token_ts: deque = deque()

    def _purge(self):
        cutoff = time.time() - 60.0
        while self._req_ts   and self._req_ts[0] < cutoff:   self._req_ts.popleft()
        while self._token_ts and self._token_ts[0][0] < cutoff: self._token_ts.popleft()

    def check(self, n_tokens: int = 0) -> Tuple[bool, str]:
        self._purge()
        if len(self._req_ts) >= self.rpm:                    return False, 'RPM exceeded'
        if sum(t for _, t in self._token_ts) + n_tokens > self.tpm: return False, 'TPM exceeded'
        return True, 'ok'

    def record(self, n_tokens: int):
        now = time.time()
        self._req_ts.append(now)
        self._token_ts.append((now, n_tokens))

class CircuitBreaker:
    def __init__(self, failure_threshold: int = 5, recovery_timeout_s: float = 30.0):
        self.failure_threshold  = failure_threshold
        self.recovery_timeout_s = recovery_timeout_s
        self._failures = 0
        self._open_at: Optional[float] = None
        self._state = 'closed'

    def call(self, fn, *args, **kwargs):
        now = time.time()
        if self._state == 'open':
            if now - self._open_at >= self.recovery_timeout_s: self._state = 'half-open'
            else: raise RuntimeError('Circuit open')
        try:
            result = fn(*args, **kwargs)
            self._failures = 0; self._state = 'closed'
            return result
        except Exception:
            self._failures += 1
            if self._failures >= self.failure_threshold:
                self._state = 'open'; self._open_at = now
            raise

    @property
    def state(self) -> str: return self._state

print('Rate limiter and circuit breaker defined.')


In [ ]:
# ── 4. Mock LLM backend ────────────────────────────────────────────────────
MODEL_COSTS = {
    'claude-opus-4-6':   {'input': 0.015,   'output': 0.075},
    'claude-sonnet-4-6': {'input': 0.003,   'output': 0.015},
    'claude-haiku-4-5':  {'input': 0.00025, 'output': 0.00125},
}

def compute_cost(model, in_tok, out_tok):
    r = MODEL_COSTS.get(model, {'input': 0.001, 'output': 0.002})
    return (in_tok * r['input'] + out_tok * r['output']) / 1000.0

@dataclass
class LLMResponse:
    content: str; model: str
    input_tokens: int; output_tokens: int
    latency_ms: float; cost_usd: float
    from_cache: bool = False

class MockLLMBackend:
    def __init__(self, failure_rate: float = 0.0, base_latency_ms: float = 1200.0):
        self.failure_rate    = failure_rate
        self.base_latency_ms = base_latency_ms
        self._rng            = np.random.default_rng(7)

    def complete(self, prompt: str, model: str = 'claude-sonnet-4-6',
                 max_tokens: int = 512) -> LLMResponse:
        if self._rng.random() < self.failure_rate:
            raise RuntimeError('Backend error')
        in_tok  = max(50, int(len(prompt.split()) * 1.3))
        out_tok = int(self._rng.integers(80, min(max_tokens, 400)))
        latency = float(max(100.0, self._rng.normal(self.base_latency_ms, self.base_latency_ms * 0.2)))
        return LLMResponse(
            content=f'[MOCK: {out_tok} tokens about {prompt[:40]}...]',
            model=model, input_tokens=in_tok, output_tokens=out_tok,
            latency_ms=latency, cost_usd=compute_cost(model, in_tok, out_tok)
        )

print('Mock backend defined.')


In [ ]:
# ── 5. Production gateway ──────────────────────────────────────────────────
@dataclass
class GatewayMetrics:
    total_requests: int   = 0
    cache_hits:     int   = 0
    cache_misses:   int   = 0
    rate_limited:   int   = 0
    circuit_broken: int   = 0
    backend_errors: int   = 0
    total_cost_usd: float = 0.0
    saved_cost_usd: float = 0.0
    latencies_ms:   List[float] = field(default_factory=list)

    def summary(self) -> Dict:
        lats = self.latencies_ms
        return {
            'total_requests': self.total_requests,
            'cache_hit_rate': self.cache_hits / max(1, self.total_requests),
            'rate_limited':   self.rate_limited,
            'circuit_broken': self.circuit_broken,
            'backend_errors': self.backend_errors,
            'total_cost_usd': round(self.total_cost_usd, 5),
            'saved_cost_usd': round(self.saved_cost_usd, 5),
            'latency_p50':    round(float(np.percentile(lats, 50)), 1) if lats else 0,
            'latency_p95':    round(float(np.percentile(lats, 95)), 1) if lats else 0,
            'latency_p99':    round(float(np.percentile(lats, 99)), 1) if lats else 0,
        }

class ProductionGateway:
    def __init__(self, backend: MockLLMBackend,
                 cache_ttl_s: float = 300.0, rpm: int = 200, tpm: int = 200_000):
        self.backend         = backend
        self.cache           = LLMCache(max_size=500, ttl_s=cache_ttl_s)
        self.rate_limiter    = RateLimiter(rpm=rpm, tpm=tpm)
        self.circuit_breaker = CircuitBreaker(failure_threshold=5, recovery_timeout_s=5.0)
        self.prompt_mgr      = PromptVersionManager()
        self.metrics         = GatewayMetrics()

    def complete(self, prompt_name: str, model: str = 'claude-sonnet-4-6',
                 max_tokens: int = 512, **kwargs) -> LLMResponse:
        t_start = time.perf_counter()
        self.metrics.total_requests += 1
        pv     = self.prompt_mgr.select(prompt_name)
        prompt = pv.render(**kwargs)
        est_tokens = max(50, len(prompt.split()) * 2)
        allowed, reason = self.rate_limiter.check(est_tokens)
        if not allowed:
            self.metrics.rate_limited += 1
            raise RuntimeError(f'Rate limited: {reason}')
        cached = self.cache.get(prompt, model, max_tokens=max_tokens)
        if cached:
            self.metrics.cache_hits += 1
            self.metrics.saved_cost_usd += compute_cost(model, cached.input_tokens, cached.output_tokens)
            resp = LLMResponse(content=cached.response, model=model,
                               input_tokens=cached.input_tokens, output_tokens=cached.output_tokens,
                               latency_ms=(time.perf_counter() - t_start) * 1000,
                               cost_usd=0.0, from_cache=True)
            self.metrics.latencies_ms.append(resp.latency_ms)
            return resp
        self.metrics.cache_misses += 1
        try:
            resp = self.circuit_breaker.call(self.backend.complete, prompt, model, max_tokens)
            self.rate_limiter.record(resp.input_tokens + resp.output_tokens)
            self.cache.put(prompt, model, resp.content, resp.input_tokens, resp.output_tokens,
                           max_tokens=max_tokens)
            self.metrics.total_cost_usd += resp.cost_usd
            resp.latency_ms = (time.perf_counter() - t_start) * 1000
            self.metrics.latencies_ms.append(resp.latency_ms)
            return resp
        except RuntimeError as e:
            if 'Circuit open' in str(e): self.metrics.circuit_broken += 1
            else: self.metrics.backend_errors += 1
            raise

print('ProductionGateway defined.')


## End-to-End Stress Test

400 requests through the full stack: mixed prompts, cache warmup, failure injection to trip the circuit breaker, then recovery.

In [ ]:
random.seed(42); np.random.seed(42)

backend = MockLLMBackend(failure_rate=0.0, base_latency_ms=1100)
gw      = ProductionGateway(backend, cache_ttl_s=600, rpm=500, tpm=2_000_000)

gw.prompt_mgr.register('summarize', PromptVersion('v1', 'Summarize: {text}', 0.5))
gw.prompt_mgr.register('summarize', PromptVersion('v2', 'Key points from: {text}', 0.5))
gw.prompt_mgr.register('qa', PromptVersion('v1', 'Answer: {question}  Context: {context}', 1.0))

TEXTS = [
    'The transformer architecture uses self-attention to process sequences in parallel.',
    'Retrieval-augmented generation combines dense retrieval with generative models.',
    'LoRA reduces fine-tuning costs by learning low-rank weight delta matrices.',
    'Speculative decoding uses a draft model to propose tokens verified by a large model.',
    'KV caching stores attention tensors to avoid recomputation across turns.',
]

errors = []

# Phase 1: 300 normal requests
for i in range(300):
    text = random.choice(TEXTS)
    try:
        if i % 3 == 0: gw.complete('qa', question='What is this?', context=text)
        else:          gw.complete('summarize', text=text)
    except Exception as e: errors.append(str(e)[:60])

# Phase 2: inject failures to trip breaker
gw.backend = MockLLMBackend(failure_rate=1.0)
for i in range(20):
    try: gw.complete('summarize', text=TEXTS[0])
    except Exception as e: errors.append(str(e)[:60])

# Phase 3: restore
gw.backend = backend
for i in range(80):
    try: gw.complete('summarize', text=random.choice(TEXTS))
    except Exception as e: errors.append(str(e)[:60])

m = gw.metrics.summary()
print('=== Stress Test Results ===')
print(f'  Total requests  : {m["total_requests"]}')
print(f'  Cache hit rate  : {m["cache_hit_rate"]*100:.1f}%')
print(f'  Rate limited    : {m["rate_limited"]}')
print(f'  Circuit broken  : {m["circuit_broken"]}')
print(f'  Backend errors  : {m["backend_errors"]}')
print(f'  Total cost      : ${m["total_cost_usd"]:.5f}')
print(f'  Saved by cache  : ${m["saved_cost_usd"]:.5f}')
print(f'  Latency p50/p95/p99: {m["latency_p50"]:.0f}/{m["latency_p95"]:.0f}/{m["latency_p99"]:.0f} ms')
print(f'\nUnique error types:')
seen = set()
for e in errors:
    key = e[:40]
    if key not in seen: print(f'  {e}'); seen.add(key)


## Production Runbook Export

In [ ]:
runbook = {
    'snapshot_time': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'service':       'llm-gateway',
    'health': {
        'status':          'degraded' if m['backend_errors'] > 10 else 'healthy',
        'circuit_breaker': gw.circuit_breaker.state,
    },
    'traffic': {
        'total_requests':  m['total_requests'],
        'cache_hit_rate':  round(m['cache_hit_rate'], 4),
        'error_rate':      round((m['backend_errors'] + m['circuit_broken']) / max(1, m['total_requests']), 4),
    },
    'latency_ms': {'p50': m['latency_p50'], 'p95': m['latency_p95'], 'p99': m['latency_p99']},
    'cost': {
        'total_usd':          m['total_cost_usd'],
        'saved_by_cache_usd': m['saved_cost_usd'],
    },
    'slo_checks': {
        'p95_under_5s':          m['latency_p95'] < 5000,
        'error_rate_under_1pct': (m['backend_errors'] + m['circuit_broken']) / max(1, m['total_requests']) < 0.01,
        'cache_over_20pct':      m['cache_hit_rate'] > 0.20,
    },
}
print(json.dumps(runbook, indent=2))
print('\nSLO Summary:')
for check, passed in runbook['slo_checks'].items():
    print(f'  [{"PASS" if passed else "FAIL"}] {check}')


## Key Takeaways

- The gateway is the single choke point: rate limiting, circuit breaking, caching, and observability all happen here — callers stay simple
- Cache keys are deterministic hashes of `(prompt, model, params)` — the same logical request from any caller hits the cache
- Circuit breakers need short recovery windows in LLM systems; backends flap in seconds, not minutes
- Prompt versioning with slot weights enables safe A/B rollouts without any flag infrastructure
- SLO checks in the runbook export give on-call engineers an instant pass/fail view — no interpreting raw numbers required